In [1]:
import numpy as np
from tqdm import tqdm
from games.kuhn import KuhnPoker
from agents.counterfactualregret_t import CounterFactualRegret
from agents.ismcts import ISMCTS
from agents.minimax import MiniMax
from agents.agent_random import RandomAgent

In [2]:
g = KuhnPoker(initial_player=0)

In [3]:
my_agents = {
    g.agents[0]: CounterFactualRegret(game=g, agent=g.agents[0]),
    #g.agents[0]: RandomAgent(game=g, agent=g.agents[1]),
    #g.agents[1]: ISMCTS(game=g, agent=g.agents[1], simulations=200, rollouts=10, rollout_iterations=100),
    g.agents[1]: CounterFactualRegret(game=g, agent=g.agents[1])
}

In [58]:
g.reset()
while not g.done():
    g.render()
    print(f"Agent {g.agent_selection}")
    action = my_agents[g.agent_selection].action()
    print(f"Action {action} - move {g.action_move(action)}")
    g.step(action)
g.render()
for agent in g.agents:
    print(f"Reward {agent} = {g.reward(agent)}")

agent_0 Q 
agent_1 K 
Agent agent_0
Action 0 - move p
agent_0 Q p
agent_1 K p
Agent agent_1
Action 1 - move b
agent_0 Q pb
agent_1 K pb
Agent agent_0
Action 0 - move p
agent_0 Q pbp
agent_1 K pbp
Reward agent_0 = -1
Reward agent_1 = 1


In [71]:
train_iterations = 1000

for agent in g.agents:
    print('Training agent ' + agent)
    if hasattr(my_agents[agent], 'train'):
        my_agents[agent].train(train_iterations)
        print(dict(map(lambda n: (n, np.round(my_agents[agent].node_dict[n].policy(), 3)), my_agents[agent].node_dict.keys())))

Training agent agent_0


100%|██████████| 1000/1000 [00:03<00:00, 252.75it/s]


{'2': array([0.145, 0.855]), '1p': array([1., 0.]), '2pb': array([0., 1.]), '1b': array([0.66, 0.34]), '0p': array([0.732, 0.268]), '0b': array([1., 0.]), '1': array([0.999, 0.001]), '1pb': array([0.413, 0.587]), '2p': array([0.003, 0.997]), '2b': array([0., 1.]), '0': array([0.692, 0.308]), '0pb': array([1., 0.])}
Training agent agent_1


100%|██████████| 1000/1000 [00:03<00:00, 277.17it/s]

{'2': array([0.5, 0.5]), '0p': array([0.682, 0.318]), '2pb': array([0., 1.]), '0b': array([1., 0.]), '1': array([0.992, 0.008]), '1pb': array([0.445, 0.555]), '0': array([0.789, 0.211]), '2p': array([0., 1.]), '0pb': array([1., 0.]), '2b': array([0., 1.]), '1p': array([0.992, 0.008]), '1b': array([0.639, 0.361])}


In [86]:
cum_rewards = dict(map(lambda agent: (agent, 0.), g.agents))
niter = 10000
for _ in tqdm(range(niter)):
    g.reset()
    #g.render()
    turn = 0
    while not g.done():
        #print('Turn: ', turn)
        #print('\tPlayer: ', g.agent_selection)
        #print('\tObservation: ', g.observe(g.agent_selection))
        a = my_agents[g.agent_selection].action()
        #print('\tAction: ', g._moves[a])
        g.step(action=a)
        turn += 1
    #print('Rewards: ', g.rewards)
    for agent in g.agents:
        cum_rewards[agent] += g.rewards[agent]
print('Average rewards:', dict(map(lambda agent: (agent, cum_rewards[agent]/niter), g.agents)))


100%|██████████| 10000/10000 [00:02<00:00, 4768.92it/s]

Average rewards: {'agent_0': -0.0342, 'agent_1': 0.0342}
